# Intra and Inter-image Clustering on Echocardiographic Frames

In this lab you receive grayscale ultrasound DICOMs where frames carry **unknown degradations** (impulse noise, Gaussian noise, contrast drift, blur, speckle). This notebook builds a controlled, labelled dataset by perturbing CAMUS clean frames with a small set of noise types, extract a handful of **hand-crafted statistical features**, and cluster. You will later apply the same pipeline unsupervised to the inhouse frames.


## 1. Setup


In [ ]:
!pip install pydicom

In [ ]:
import random
import warnings
from pathlib import Path
import os

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import pydicom

from scipy.stats import skew, kurtosis
from scipy.ndimage import laplace

from sklearn.preprocessing import StandardScaler
from sklearn.decomposition import PCA
from sklearn.cluster import HDBSCAN, KMeans
from sklearn.metrics import adjusted_rand_score, silhouette_score
import json
from typing import TypedDict

import math

from skimage.draw import polygon

SEED = 42
rng = np.random.default_rng(SEED)
random.seed(SEED)
warnings.filterwarnings("ignore", category=UserWarning)

In [ ]:
try:
    from google.colab import drive
    IN_COLAB = True
except ImportError:
    IN_COLAB = False

if IN_COLAB:
    # Mount the drive containing the images
    drive.mount('/content/drive')

    # List contents (IPython magic command)
    !ls

In [ ]:
if IN_COLAB:
    # Change the working to the current notebook's directory
    %cd /content/drive/MyDrive/
else:
    # Ask the user to fill in the base data folder path
    base_data_folder = input("Please enter the path to the base data folder path: ")
    os.chdir(base_data_folder)

In [ ]:
if IN_COLAB:
    !ls
else:
    files = os.listdir(base_data_folder)
    for f in files:
        print(f)

In [ ]:
def pie_slice_mask(
    apex: tuple[float, float],
    LB: tuple[float, float],
    RB: tuple[float, float],
    shape: tuple[int, int],
) -> np.ndarray:
    """Return a boolean pie-slice mask defined by three (y, x) landmarks.

    Parameters
    ----------
    apex:
        Tip of the ultrasound cone ``(y, x)``.
    LB:
        Left-bottom corner of the cone ``(y, x)``.
    RB:
        Right-bottom corner of the cone ``(y, x)``.
    shape:
        ``(H, W)`` of the target image.

    Returns
    -------
    bool ndarray of ``shape`` — ``True`` inside the pie slice.

    Notes
    -----
    The arc boundary interpolates the radius linearly from ``dist(apex, LB)``
    at angle ``θ_L`` to ``dist(apex, RB)`` at angle ``θ_R``, so the arc
    passes exactly through both LB and RB. The apex is used as clicked with
    no correction applied.
    """
    apex_y, apex_x = float(apex[0]), float(apex[1])
    lb_y, lb_x = float(LB[0]), float(LB[1])
    rb_y, rb_x = float(RB[0]), float(RB[1])

    theta_L = math.atan2(lb_y - apex_y, lb_x - apex_x)
    theta_R = math.atan2(rb_y - apex_y, rb_x - apex_x)
    r_L = math.hypot(lb_y - apex_y, lb_x - apex_x)
    r_R = math.hypot(rb_y - apex_y, rb_x - apex_x)

    th_min, th_max = min(theta_L, theta_R), max(theta_L, theta_R)
    r_at_min, r_at_max = (r_L, r_R) if theta_L <= theta_R else (r_R, r_L)

    arc_angles = np.linspace(th_min, th_max, num=200)
    span = th_max - th_min
    t = (arc_angles - th_min) / span if span > 0 else np.zeros(200)
    arc_radii = r_at_min + t * (r_at_max - r_at_min)

    arc_rows = apex_y + arc_radii * np.sin(arc_angles)
    arc_cols = apex_x + arc_radii * np.cos(arc_angles)

    poly_rows = np.concatenate(([apex_y], arc_rows))
    poly_cols = np.concatenate(([apex_x], arc_cols))

    rr, cc = polygon(poly_rows, poly_cols, shape=shape[:2])
    mask = np.zeros(shape[:2], dtype=bool)
    mask[rr, cc] = True
    return mask


class SectorEntry(TypedDict):
    apex: list[float]  # [y, x]
    LB: list[float]  # [y, x]
    RB: list[float]  # [y, x]
    machine: str
    shape: list[int]  # [H, W]


def _normalize_key(raw_rel_path: str | Path) -> str:
    """Strip a leading ``./`` or ``.\\`` and force POSIX separators."""
    s = str(raw_rel_path).replace("\\", "/")
    if s.startswith("./"):
        s = s[2:]
    return s


def load_sector_cache(json_path: Path) -> dict[str, SectorEntry]:
    """Load the JSON sector cache, normalising keys to a leading-``./``-free form.

    Raises ``FileNotFoundError`` with a hint to run the annotator CLI when
    the file is missing.
    """
    json_path = Path(json_path)
    if not json_path.exists():
        raise FileNotFoundError(
            f"Sector cache not found: {json_path}. "
            "Run scripts/annotate_sectors_cli.py to generate it."
        )
    raw = json.loads(json_path.read_text())
    return {_normalize_key(k): v for k, v in raw.items()}


def build_sector_mask(
    cache: dict[str, SectorEntry],
    raw_rel_path: str | Path,
    target_shape: tuple[int, int],
) -> np.ndarray:
    """Return the boolean sector mask for ``raw_rel_path`` at ``target_shape``.

    The entry's ``(apex, LB, RB)`` are rescaled from the stored raw ``shape``
    to ``target_shape`` by a pure per-axis linear factor (the only transform
    :func:`src.io._resample.resample_to_spacing` applies). The pie slice is
    then rasterised at ``target_shape``.

    Raises ``KeyError`` when ``raw_rel_path`` is not present in the cache.
    """
    key = _normalize_key(raw_rel_path)
    if key not in cache:
        raise KeyError(
            f"No sector annotation for {key!r}. "
            "Re-run scripts/annotate_sectors_cli.py so every frame is covered."
        )
    entry = cache[key]

    src_h, src_w = int(entry["shape"][0]), int(entry["shape"][1])
    dst_h, dst_w = int(target_shape[0]), int(target_shape[1])
    sy = dst_h / src_h
    sx = dst_w / src_w

    apex = (entry["apex"][0] * sy, entry["apex"][1] * sx)
    LB = (entry["LB"][0] * sy, entry["LB"][1] * sx)
    RB = (entry["RB"][0] * sy, entry["RB"][1] * sx)

    return pie_slice_mask(apex, LB, RB, (dst_h, dst_w))

## 2. What is clustering?

Clustering groups samples by similarity **without labels**. Two broad families:

- **Partitional methods** — e.g. **KMeans**. They need the number of clusters `K` in advance and assume roughly convex clusters.
- **Density-based methods** — e.g. **HDBSCAN**. They infer `K` from local density and label low-density points as outliers (`-1`). Robust to irregular shapes and to a background of unassigned noise.

We will use HDBSCAN as the primary clusterer and compare against KMeans(4), which can exploit our known number of synthetic classes.


## 3. Dataset: CAMUS clean frames

We load clean frames from the student release at `Datasets/lab-bioimmagini-data/release/camus/<split>/clean/*.dcm`. Each DICOM is a `uint8` grayscale image which we normalise to `float32` in `[0, 1]`.


**Sector-mask parity with the production pipeline.** We use the pre-computed sector mask from `camus_cache.json` via `src.masking.load_sector_cache` and `src.masking.build_sector_mask` — the same cache the production pipeline consumes. All perturbations and statistics in this notebook operate **inside this mask**, so the showcase mirrors the inhouse pipeline students will encounter: the canonical kernels in `src/perturbations/kernels.py` are sector-only, and so are the features here.

**Local sample cache.** To keep re-runs fast and deterministic we cache a 200-frame subsample on disk at `SAMPLE_DIR` (see the next cell). Only the **clean DICOMs** and a pruned sector JSON are persisted; perturbed images stay in RAM and are regenerated from `SEED = 42` every run, so students can vary the seed cheaply without losing the base sample.

In [ ]:
import json
import shutil
import sys

SAMPLE_DIR = Path("./clustering_sample")
SAMPLE_IMAGES = SAMPLE_DIR / "images"
SAMPLE_CACHE_JSON = SAMPLE_DIR / "sector_cache.json"
N_FRAMES = 200


def cache_key_for(dicom_path: Path) -> str:
    """Map a release-clean DICOM path to its camus_cache.json key.

    Release filenames look like `patient0273_2CH_ED.dcm`; cache keys look like
    `database_nifti/patient0273/patient0273_2CH_ED.nii.gz` (normalised by
    `load_sector_cache` to drop any leading `./`).
    """
    stem = Path(dicom_path).stem
    patient = stem.split("_", 1)[0]
    return f"database_nifti/{patient}/{stem}.nii.gz"


sample_paths = sorted(SAMPLE_IMAGES.glob("*.dcm"))
SECTOR_CACHE = load_sector_cache(SAMPLE_CACHE_JSON)
print(f"Sample size: {len(sample_paths)}; sector cache entries: {len(SECTOR_CACHE)}.")
print(f"Example cache key: {next(iter(SECTOR_CACHE))}")

In [ ]:
def load_frame(path):
    """Load a CAMUS DICOM frame as float32 in [0, 1]."""
    arr = pydicom.dcmread(str(path)).pixel_array.astype(np.float32)
    return arr / 255.0


frames = [load_frame(p) for p in sample_paths]
masks = [
    build_sector_mask(SECTOR_CACHE, cache_key_for(p), f.shape)
    for p, f in zip(sample_paths, frames)
]
print(
    f"Loaded {len(frames)} frames; example shape = {frames[0].shape}, dtype = {frames[0].dtype}"
)
_cov = np.mean([m.mean() for m in masks])
print(f"Mean sector coverage across the sample: {_cov:.1%} of pixels per frame.")

# Preview: top row = frames, bottom-left shows one frame with its sector mask outlined.
fig, axes = plt.subplots(4, 4, figsize=(10, 10))
for ax, img in zip(axes.flat, frames[:16]):
    ax.imshow(img, cmap="gray", vmin=0, vmax=1)
    ax.axis("off")
plt.suptitle("Sample of clean CAMUS frames", y=0.92)
plt.show()

fig, ax = plt.subplots(1, 1, figsize=(5, 5))
ax.imshow(frames[0], cmap="gray", vmin=0, vmax=1)
ax.imshow(np.where(masks[0], 1.0, np.nan), cmap="autumn", alpha=0.25, vmin=0, vmax=1)
ax.set_title(f"Sector mask overlay (coverage={masks[0].mean():.1%})")
ax.axis("off")
plt.show()

## 4. Pixel-level Intra-image Clustering

Clustering at pixel level: every **pixel** is a sample and the clusters are **image regions**. This is the simplest form of unsupervised image segmentation.

### Pixel feature vector

Each pixel $p$ is described by its intensity and its position:

$$\mathbf{x}_p = \Big(I(p),\ \lambda\,\tfrac{y_p}{H},\ \lambda\,\tfrac{x_p}{W}\Big)$$

- **Intensity only** would group pixels by brightness wherever they are: two dark structures far apart end up in the same cluster, and speckle makes the result look like salt and pepper.
- **Adding the position** pushes the clusters to be spatially compact, i.e. connected regions.
- **Why rescale the coordinates?** Intensity lives in $[0, 1]$ while pixel coordinates span hundreds of units, so unscaled coordinates would dominate every distance. We divide by the image size, so $y/H$ and $x/W$ also live in $[0, 1]$, and we use a weight $\lambda$ (`POSITION_WEIGHT`) to tune the trade-off: $\lambda = 0$ clusters on intensity only; a large $\lambda$ ignores the intensity and just tiles the image.

### From image to matrix, and back (the reshape)

scikit-learn expects a matrix of shape `(n_samples, n_features)`, while an image is `(H, W)`.

1. stack the maps intensity, $y$, $x$ into an array `(h, w, 3)`;
2. `reshape(-1, 3)` gives `(h*w, 3)`, **one row per pixel**;
3. cluster: we get one label per row;
4. put the labels back in an `(h, w)` **label image** (a `reshape(h, w)`).

In an ultrasound frame the area outside the cone is black and carries no information, so clustering it would only create a trivial "background" cluster. We therefore keep only the rows inside the sector mask (a boolean vector, `in_sector`) and, when going back to the image shape, we fill the other positions with a sentinel value.

### Two practical choices

- **Downsampling.** A frame has hundreds of thousands of pixels, and HDBSCAN needs memory and time that grow quickly with the number of samples. We average the frame in blocks of `DOWNSAMPLE` x `DOWNSAMPLE` pixels (which also attenuates speckle), cluster at this lower resolution, and bring the labels back to full resolution by repeating each label over its block (nearest neighbour).
- **Parameters.** KMeans needs `K`. For HDBSCAN we express `min_cluster_size` as a fraction of the sector pixels, so that it does not change when `DOWNSAMPLE` changes.

This is a demonstration of the mechanics on a clean frame, where there is no ground truth, and the clusters are regions of similar brightness and position, not anatomy.

In [ ]:
from skimage.transform import downscale_local_mean

FRAME_IDX = 0  # which clean frame
DOWNSAMPLE = 4  # block-averaging factor (1 = full resolution: slow for HDBSCAN)
POSITION_WEIGHT = (
    0.5  # lambda: weight of the (y, x) coordinates relative to the intensity
)

img_full = frames[FRAME_IDX]
mask_full = masks[FRAME_IDX]

# 1) Lower resolution: mean pooling of the frame and of its sector mask
img_s = downscale_local_mean(img_full, (DOWNSAMPLE, DOWNSAMPLE)).astype(np.float32)
mask_s = (
    downscale_local_mean(mask_full.astype(np.float32), (DOWNSAMPLE, DOWNSAMPLE)) > 0.5
)
h, w = img_s.shape

# 2) Pixel coordinates, rescaled to [0, 1] like the intensity
yy, xx = np.mgrid[0:h, 0:w]
y_norm, x_norm = yy / (h - 1), xx / (w - 1)

# 3) RESHAPE: three (h, w) maps -> (h, w, 3) -> (h*w, 3), one row per pixel
features = np.stack(
    [img_s, POSITION_WEIGHT * y_norm, POSITION_WEIGHT * x_norm], axis=-1
)
X_pix = features.reshape(-1, 3)
in_sector = mask_s.reshape(-1)  # True for the rows that belong to the ultrasound cone
X_sector = X_pix[in_sector]  # the samples that we actually cluster

print(f"Frame {img_full.shape} -> low resolution {img_s.shape}")
print(f"Pixels inside the sector: {len(X_sector):,} of {h * w:,}")
print(f"Feature matrix: {X_sector.shape}  (n_pixels x [intensity, y, x])")
# Show the low-resolution frame with the sector mask overlayed
fig, axes = plt.subplots(1, 1, figsize=(12, 5))
axes.imshow(img_s, cmap="gray", vmin=0, vmax=1)
# axes.imshow(mask_s, cmap="Reds", alpha=0.5)

In [ ]:
OUTSIDE = (
    -2
)  # sentinel for pixels outside the cone (HDBSCAN noise keeps its usual label, -1)


def labels_to_image(labels, in_sector, shape, outside=OUTSIDE):
    """Inverse of the reshape: scatter the labels of the sector pixels back into an (h, w) label image."""
    flat = np.full(in_sector.size, outside, dtype=int)
    flat[in_sector] = labels
    return flat.reshape(shape)


# --- KMeans: the number of clusters must be chosen in advance
K = 4
km_pix = KMeans(n_clusters=K, n_init=10, random_state=SEED).fit(X_sector)
label_km = labels_to_image(km_pix.labels_, in_sector, (h, w))

# --- HDBSCAN: no K; min_cluster_size is a fraction of the sector pixels
MIN_CLUSTER_FRAC = 0.02
min_cluster_size = max(20, int(MIN_CLUSTER_FRAC * len(X_sector)))
hdb_pix = HDBSCAN(min_cluster_size=min_cluster_size, min_samples=10, copy=True).fit(
    X_sector
)
label_hdb = labels_to_image(hdb_pix.labels_, in_sector, (h, w))

n_hdb = len(set(hdb_pix.labels_.tolist()) - {-1})
print(f"KMeans : {K} clusters (chosen by us)")
print(
    f"HDBSCAN: {n_hdb} clusters found, {(hdb_pix.labels_ == -1).sum()} noise pixels "
    f"(min_cluster_size = {min_cluster_size})"
)

Each cluster is a **boolean mask** of the image, `label_img == k`. We bring the label images back to the full resolution of the frame and use the masks to isolate each region: the left panel overlays all the clusters, the other panels keep only the pixels of one cluster (`np.where(cluster_mask, img, 0)`). For HDBSCAN, the noise pixels (label `-1`) get their own panel.

In [ ]:
import numpy as np
import matplotlib.pyplot as plt
import math


def upsample_labels(label_img, factor, out_shape, mask_full, outside):
    """Nearest-neighbour upsampling of a label image to the full-resolution frame shape."""
    big = np.repeat(np.repeat(label_img, factor, axis=0), factor, axis=1)
    out = np.full(out_shape, outside, dtype=label_img.dtype)
    hh, ww = min(out_shape[0], big.shape[0]), min(out_shape[1], big.shape[1])
    out[:hh, :ww] = big[:hh, :ww]
    out[~mask_full] = outside  # keep exactly the original sector
    return out


def plot_all_clusters(subfig, img, label_full, name, outside):
    """Helper to plot the single overlay image in a specific subfigure."""
    ax = subfig.subplots(1, 1)
    ids = [int(k) for k in np.unique(label_full) if k != outside]
    cmap = plt.get_cmap("tab10")
    colour = {k: ((0.7, 0.7, 0.7) if k == -1 else cmap(k % 10)[:3]) for k in ids}

    overlay = np.zeros(label_full.shape + (4,), dtype=np.float32)
    for k in ids:
        overlay[label_full == k] = (*colour[k], 0.45)

    ax.imshow(img, cmap="gray", vmin=0, vmax=1)
    ax.imshow(overlay)
    ax.set_title(f"{name}\nAll Clusters", fontsize=14, fontweight="bold")
    ax.axis("off")


def plot_individual_clusters(subfig, img, label_full, mask_full, name, outside):
    """Helper to plot individual cluster masks in a 2-column grid in a specific subfigure."""
    ids = [int(k) for k in np.unique(label_full) if k != outside]
    n_clusters = len(ids)

    if n_clusters == 0:
        return

    cols = 2
    rows = math.ceil(n_clusters / cols)

    axes = subfig.subplots(rows, cols)
    subfig.suptitle(f"{name} Individual Clusters", fontsize=14, fontweight="bold")

    # Flatten axes for easy iteration
    axes = np.atleast_1d(axes).flatten()

    for i, (ax, k) in enumerate(zip(axes, ids)):
        cluster_mask = label_full == k
        ax.imshow(np.where(cluster_mask, img, 0.0), cmap="gray", vmin=0, vmax=1)
        title_str = "Noise (-1)" if k == -1 else f"Cluster {k}"
        pct = cluster_mask.sum() / mask_full.sum()
        ax.set_title(f"{title_str}\n{pct:.1%} of sector")
        ax.axis("off")

    # Turn off axes for any empty slots if n_clusters is odd
    for j in range(n_clusters, len(axes)):
        axes[j].axis("off")


def show_combined_segmentations(img, label_km, label_hdb, mask_full, k_val, outside):
    """
    Creates a single 2x2 master figure:
    (0,0): KMeans all clusters      | (0,1): HDBSCAN all clusters
    ----------------------------------------------------------------
    (1,0): KMeans individual grid   | (1,1): HDBSCAN individual grid
    """
    fig = plt.figure(figsize=(18, 14), constrained_layout=True)
    # Create 2x2 quadrants
    subfigs = fig.subfigures(2, 2, wspace=0.05, hspace=0.05)

    # Top-Left (0,0): KMeans all clusters
    plot_all_clusters(subfigs[0, 0], img, label_km, f"KMeans (K={k_val})", outside)

    # Top-Right (0,1): HDBSCAN all clusters
    plot_all_clusters(subfigs[0, 1], img, label_hdb, "HDBSCAN", outside)

    # Bottom-Left (1,0): KMeans individual clusters (2 images per row)
    plot_individual_clusters(
        subfigs[1, 0], img, label_km, mask_full, f"KMeans (K={k_val})", outside
    )

    # Bottom-Right (1,1): HDBSCAN individual clusters (2 images per row)
    plot_individual_clusters(
        subfigs[1, 1], img, label_hdb, mask_full, "HDBSCAN", outside
    )

    plt.show()


label_km_full = upsample_labels(
    label_km, DOWNSAMPLE, img_full.shape, mask_full, OUTSIDE
)
label_hdb_full = upsample_labels(
    label_hdb, DOWNSAMPLE, img_full.shape, mask_full, OUTSIDE
)

show_combined_segmentations(
    img_full, label_km_full, label_hdb_full, mask_full, K, OUTSIDE
)

In [ ]:
# Summary of each cluster: size, mean intensity and mean (normalised) position of its pixels.
# Same row order as X_sector, so the labels can be attached directly.
pix = pd.DataFrame(
    {"intensity": img_s[mask_s], "y": y_norm[mask_s], "x": x_norm[mask_s]}
)
assert np.allclose(pix["intensity"].values, X_sector[:, 0])
pix["KMeans"] = km_pix.labels_
pix["HDBSCAN"] = hdb_pix.labels_

for method in ("KMeans", "HDBSCAN"):
    print(
        f"\n{method}: size, mean intensity and mean position of each cluster (-1 = noise)"
    )
    display(
        pix.groupby(method)
        .agg(
            n_pixels=("intensity", "size"),
            mean_intensity=("intensity", "mean"),
            mean_y=("y", "mean"),
            mean_x=("x", "mean"),
        )
        .round(3)
    )

## 5. Perturbation playground: three noise kernels

We apply **one** of the following perturbations to each frame — or leave it clean. This creates a labeled 4-class dataset for the showcase. The formulas mirror the canonical kernels and severities seen in the previous labs.

In [ ]:
def salt_pepper(img, mask, fraction, ratio=0.5, rng=rng):
    """Impulse noise applied only inside `mask`: a fraction of sector pixels is split
    into salt (1.0) and pepper (0.0). Outside-mask pixels pass through unchanged."""
    out = img.copy()
    sector_idx = np.flatnonzero(mask.ravel())
    n = int(round(fraction * sector_idx.size))
    if n == 0:
        return out
    chosen = rng.choice(sector_idx, size=n, replace=False)
    salt_count = int(round(ratio * n))
    out.flat[chosen[:salt_count]] = 1.0
    out.flat[chosen[salt_count:]] = 0.0
    return out


def gaussian_noise(img, mask, psnr_db_target, rng=rng):
    """Additive white Gaussian noise applied only inside `mask`."""
    sigma = float(np.sqrt(10.0 ** (-psnr_db_target / 10.0)))
    noise = rng.standard_normal(img.shape).astype(np.float32) * sigma
    out = img.copy()
    out[mask] = np.clip(img[mask] + noise[mask], 0.0, 1.0)
    return out


def contrast(img, mask, ln_gamma, s):
    """Gamma remap + linear contrast reduction, applied only inside `mask`.
    The mean `mu` is computed on the sector pixels so it reflects the local
    brightness distribution, matching `src/perturbations/kernels.py`."""
    gamma = float(np.exp(ln_gamma))
    out = img.copy()
    if not mask.any():
        return out
    x_pow = np.power(img, gamma, dtype=np.float32)
    mu = float(x_pow[mask].mean())
    out[mask] = (mu + s * (x_pow[mask] - mu)).astype(np.float32)
    return out

In [ ]:
NOISE_TYPES = ("clean", "salt_pepper", "gaussian_noise", "contrast")

MILD_PARAMS = {
    "salt_pepper": {"fraction": 0.005, "ratio": 0.5},
    "gaussian_noise": {"psnr_db_target": 30.0},
    "contrast": {"ln_gamma_abs": (0.10, 0.25), "s": (0.80, 0.90)},
}
MODERATE_PARAMS = {
    "salt_pepper": {"fraction": 0.02, "ratio": 0.5},
    "gaussian_noise": {"psnr_db_target": 25.0},
    "contrast": {"ln_gamma_abs": (0.25, 0.45), "s": (0.60, 0.80)},
}


def apply_perturbation(img, mask, ntype, severity, rng):
    if ntype == "clean":
        return img.copy()
    params = (MILD_PARAMS if severity == "mild" else MODERATE_PARAMS)[ntype]
    if ntype == "salt_pepper":
        return salt_pepper(img, mask, params["fraction"], params["ratio"], rng)
    if ntype == "gaussian_noise":
        return gaussian_noise(img, mask, params["psnr_db_target"], rng)
    if ntype == "contrast":
        lo, hi = params["ln_gamma_abs"]
        abs_ln = float(rng.uniform(lo, hi))
        sign = 1.0 if rng.integers(0, 2) == 0 else -1.0
        s = float(rng.uniform(*params["s"]))
        return contrast(img, mask, sign * abs_ln, s)
    raise ValueError(ntype)


# Reseed from SEED so the perturbation draw is independent of the sample bootstrap.
rng = np.random.default_rng(SEED)

records, images = [], []
for i, (frame, mask) in enumerate(zip(frames, masks)):
    ntype = str(rng.choice(NOISE_TYPES))
    severity = "clean" if ntype == "clean" else str(rng.choice(("mild", "moderate")))
    out = apply_perturbation(frame, mask, ntype, severity, rng)
    records.append({"frame_id": i, "noise_type": ntype, "severity": severity})
    images.append(out)

df = pd.DataFrame(records)
print(df["noise_type"].value_counts().rename("count").to_frame())

fig, axes = plt.subplots(1, 4, figsize=(14, 4))
for ax, ntype in zip(axes, NOISE_TYPES):
    idx = int(df.index[df["noise_type"] == ntype][0])
    ax.imshow(images[idx], cmap="gray", vmin=0, vmax=1)
    ax.set_title(f"{ntype} ({df.loc[idx, 'severity']})")
    ax.axis("off")
plt.suptitle(
    "One example per noise class (perturbations applied inside the sector)", y=1.02
)
plt.show()

## 6. Feature engineering: 9 hand-crafted statistics

Each noise type leaves a signature on first-order pixel statistics inside the ultrasound sector. We compute nine features per frame on `img[mask]`. For `laplacian_var` we run the Laplacian on the full image first and then take the variance of the masked region — this preserves local derivatives while **excluding sector-boundary gradients** that would otherwise dominate.


| # | Feature | Fingerprints | Why |
|---|---|---|---|
| 1 | `mean` | contrast | gamma + rescale shifts the sector mean; S&P and Gaussian are mean-neutral once computed over sector pixels only |
| 2 | `std` | contrast (↓), gaussian_noise (↑), salt_pepper (↑) | $s<1$ shrinks the sector std; Gaussian additively grows it by $\sigma$; impulse noise inflates it from the tails |
| 3 | `frac_low` | salt_pepper | pepper injects pure-0 pixels inside the sector (sector pixels are rarely 0 otherwise, so this is a sharp detector) |
| 4 | `frac_high` | salt_pepper | salt injects pure-1 pixels; a strongly negative $\ln\gamma$ can also push some pixels to 1 |
| 5 | `delta_global_mean` | contrast sign | relative to the sector-pool mean; sign disambiguates $\ln\gamma$ direction |
| 6 | `delta_global_std` | gaussian_noise vs contrast | positive $\Rightarrow$ Gaussian; negative $\Rightarrow$ contrast reduction |
| 7 | `skew` | salt_pepper, contrast | S&P distorts tail symmetry; non-linear gamma also skews the sector histogram |
| 8 | `kurtosis` | salt_pepper (↑), gaussian_noise (↓) | impulse noise concentrates extreme bins; Gaussian broadens the main peak |
| 9 | `log_laplacian_var` | **the key separator**: contrast + clean ≈ -6.5, gaussian_noise ≈ -3.3, salt_pepper ≈ -2.6 | High-frequency energy detector. The raw Laplacian variance spans two orders of magnitude across classes; the log compresses it so every class occupies its own band. |

Two features (`delta_global_mean`, `delta_global_std`) are computed *relative to the sector-pool* across all 200 frames, so each image's "unusualness" is measured against the sector-only distribution.

In [ ]:
EPS = 0.02
_LOG_EPS = 1e-8


def extract_features(img, mask, global_mean, global_std):
    """Nine first-order statistics computed over sector pixels only.

    `log_laplacian_var` is `log(laplacian_var + eps)` — the log compresses the
    two-order-of-magnitude dynamic range of the raw variance so every class
    occupies its own band after standardisation.
    """
    vals = img[mask]
    m = float(vals.mean())
    s = float(vals.std())
    lap = laplace(img)
    return {
        "mean": m,
        "std": s,
        "frac_low": float((vals <= EPS).mean()),
        "frac_high": float((vals >= 1.0 - EPS).mean()),
        "delta_global_mean": m - global_mean,
        "delta_global_std": s - global_std,
        "skew": float(skew(vals)),
        "kurtosis": float(kurtosis(vals, fisher=True)),
        "log_laplacian_var": float(np.log(lap[mask].var() + _LOG_EPS)),
    }

In [ ]:
sector_pixels = np.concatenate([img[m] for img, m in zip(images, masks)])
global_mean = float(sector_pixels.mean())
global_std = float(sector_pixels.std())
print(
    f"Sector-pool global: mean={global_mean:.4f}, std={global_std:.4f} "
    f"(computed over {sector_pixels.size:,} pixels across {len(images)} frames)"
)

feat_rows = [
    extract_features(img, m, global_mean, global_std) for img, m in zip(images, masks)
]
feat_df = pd.DataFrame(feat_rows)
feat_df["noise_type"] = df["noise_type"].values

print("\nPer-class mean of each feature:")
feat_df.groupby("noise_type").mean().round(4)

## 7. Clustering the feature space

The nine features span different units and scales, so we **standardise** them before clustering — otherwise HDBSCAN's density estimate collapses onto the dominant axis.


Two clusterers side-by-side:
- **HDBSCAN** — density-based, no `K` to set, outputs an outlier bucket (`-1`).
- **KMeans(K=4)** — informed baseline since we constructed 4 classes.

PCA to 2-D is used **for visualisation only**, not for clustering.

In [ ]:
feature_cols = [c for c in feat_df.columns if c != "noise_type"]
scaler = StandardScaler().fit(feat_df[feature_cols].values)
X = scaler.transform(feat_df[feature_cols].values)
print(f"Standardised feature matrix: shape={X.shape}")

In [ ]:
y_true = df["noise_type"].values

hdb = HDBSCAN(min_cluster_size=5, min_samples=3).fit(X)
y_hdb = hdb.labels_
y_kmeans = KMeans(n_clusters=4, random_state=SEED, n_init=10).fit_predict(X)


def summary(name, labels):
    labels = np.asarray(labels)
    n_clusters = len(set(labels.tolist())) - (1 if -1 in labels else 0)
    n_noise = int((labels == -1).sum())
    return f"{name:>9}: {n_clusters} clusters, {n_noise} noise pts"


print(summary("HDBSCAN", y_hdb))
print(summary("KMeans", y_kmeans))

In [ ]:
X2 = PCA(n_components=2, random_state=SEED).fit_transform(X)


def scatter_labels(ax, X2, lab, title, cmap_name="tab10"):
    labs = np.array(lab)
    uniq = sorted(set(labs.tolist()), key=lambda v: (str(v) == "-1", str(v)))
    cmap = plt.get_cmap(cmap_name)
    for i, k in enumerate(uniq):
        m = labs == k
        if str(k) == "-1":
            ax.scatter(
                X2[m, 0], X2[m, 1], c="lightgrey", marker="x", s=22, label="noise (-1)"
            )
        else:
            ax.scatter(X2[m, 0], X2[m, 1], c=[cmap(i % cmap.N)], s=26, label=str(k))
    ax.set_title(title)
    ax.set_xticks([])
    ax.set_yticks([])
    ax.legend(loc="best", fontsize=7, markerscale=0.8, frameon=False)


fig, axes = plt.subplots(1, 3, figsize=(15, 5))
scatter_labels(axes[0], X2, y_true, "True labels")
scatter_labels(axes[1], X2, y_hdb, "HDBSCAN")
scatter_labels(axes[2], X2, y_kmeans, "KMeans (K=4)")
plt.suptitle("PCA(2) projection — colours = cluster labels", y=1.02)
plt.tight_layout()
plt.show()

## 8. Evaluation

Two kinds of metric are useful here:

- **External metrics** compare predicted clusters against known true labels. We use **ARI** (adjusted Rand index): it counts how often pairs of points land in the same cluster for both the predicted and true labelling, corrected so a random assignment scores 0 and a perfect match scores 1.
- **Internal metrics** score the geometry of the clusters without using labels. We use the **silhouette**: for each point, how much closer is it to its own cluster than to the next nearest? It ranges from -1 to +1 (higher is better).

ARI tells you *how well* a clustering matches reality but not *which* clusters absorb each true class. For that we use the **confusion matrix**.


In [ ]:
ext_df = pd.DataFrame(
    {
        "HDBSCAN": {"ARI": adjusted_rand_score(y_true, y_hdb)},
        "KMeans(4)": {"ARI": adjusted_rand_score(y_true, y_kmeans)},
    }
).T.round(3)
ext_df

In [ ]:
class_order = list(NOISE_TYPES)


def plot_cm(ax, y_true, y_pred, title):
    y_true_s = np.array([str(v) for v in y_true])
    y_pred_s = np.array([str(v) for v in y_pred])
    pred_order = sorted(set(y_pred_s.tolist()), key=lambda v: (v == "-1", v))
    cm = np.zeros((len(class_order), len(pred_order)), dtype=int)
    for i, t in enumerate(class_order):
        for j, p in enumerate(pred_order):
            cm[i, j] = int(((y_true_s == t) & (y_pred_s == p)).sum())
    ax.imshow(cm, cmap="Blues")
    ax.set_xticks(range(len(pred_order)))
    ax.set_xticklabels(pred_order, rotation=45, ha="right")
    ax.set_yticks(range(len(class_order)))
    ax.set_yticklabels(class_order)
    vmax = max(cm.max(), 1)
    for i in range(cm.shape[0]):
        for j in range(cm.shape[1]):
            ax.text(
                j,
                i,
                cm[i, j],
                ha="center",
                va="center",
                color="white" if cm[i, j] > vmax / 2 else "black",
                fontsize=9,
            )
    ax.set_title(title)
    ax.set_xlabel("predicted")
    ax.set_ylabel("true")


fig, axes = plt.subplots(1, 2, figsize=(11, 5))
plot_cm(axes[0], y_true, y_hdb, "HDBSCAN")
plot_cm(axes[1], y_true, y_kmeans, "KMeans(4)")
plt.tight_layout()
plt.show()

In [ ]:
def silhouette(X, labels):
    labels = np.asarray(labels)
    mask = labels != -1
    if mask.sum() < 2 or len(set(labels[mask].tolist())) < 2:
        return np.nan
    return silhouette_score(X[mask], labels[mask])


int_df = pd.DataFrame(
    {
        "HDBSCAN": {"silhouette": silhouette(X, y_hdb)},
        "KMeans(4)": {"silhouette": silhouette(X, y_kmeans)},
    }
).T.round(3)
int_df

## 9. Takeaways

1. **Hand-crafted features beat CNN embeddings** when the discriminative signal is first-order pixel statistics. Every one of our 9 axes has a physical meaning tied to a specific noise mechanism — centroids are directly interpretable.
2. **Density-based clustering surfaces outliers for free.** KMeans assigns every point to some cluster, so ambiguous frames get mis-assigned. HDBSCAN leaves them in the `-1` bucket, which is itself informative.
3. **High silhouette ≠ correct clusters.** KMeans wins silhouette (0.26) because it directly optimises within-cluster compactness — but its ARI is 0.07, meaning its four geometrically beautiful cells slice *across* the true classes. HDBSCAN has the lower silhouette (0.16) yet the much higher ARI (0.34) because its sub-clusters sit close to each other in Euclidean space while being semantically correct. On the unlabelled inhouse homework you will not have ARI as a sanity check — so interpret silhouette with care, and lean on centroid inspection and the fingerprint table from §5.
4. The gap between the labelled (CAMUS) showcase and the unlabelled (inhouse) homework is the gap between **external and internal metrics**.


## 10. Homework: apply the same pipeline to inhouse frames

The showcase above was labelled: *we* built the dataset by applying known perturbations. Your homework is to run the **same** methodology on the student release of the inhouse dataset, where **you do not have the labels**.

### What stays the same

- `load_frame` (inhouse DICOMs are also uint8 grayscale).
- Sector-aware feature extraction: load the sibling cache at `raw/inhouse/inhouse_cache.json` with `src.masking.load_sector_cache`, build each frame's mask with `src.masking.build_sector_mask`, then run the same `extract_features(img, mask, global_mean, global_std)`.
- `StandardScaler().fit_transform(...)` → `HDBSCAN(min_cluster_size=5, min_samples=3)` → `PCA(n_components=2)` for visualisation. These HDBSCAN defaults were tuned on the showcase; the inhouse feature-space density will differ, so sweep `(min_cluster_size, min_samples)` on your own data before locking defaults — pick the setting whose cluster count and noise fraction look reasonable, not the one that maximises silhouette (see Takeaway 3).
- Optionally mirror the `SAMPLE_DIR` cache pattern from §3 to keep inhouse re-runs fast.

### What changes

- **Global stats.** Recompute `global_mean, global_std` from inhouse **sector pixels** — do NOT reuse the CAMUS values. The acquisition device and view distribution differ.
- **No ground truth.** ARI and the confusion matrix are *not* computable. Evaluate quality only via the **silhouette** — with the caveat from Takeaway 3 that silhouette can prefer the wrong clusterer.
- **Richer label space.** Every inhouse frame has $k \in \{0, 1, 2, 3\}$ perturbations drawn uniformly from **5 kernels** (`gaussian_blur`, `contrast`, `speckle`, `gaussian_noise`, `salt_pepper`) across **3 severity bands** (`mild`, `moderate`, `severe`). Empirical release distribution (from `internal/perturbations_inhouse.csv`):

  | k | fraction |
  |---|---|
  | 0 (clean) | 15.2 % |
  | 1 | 40.1 % |
  | 2 | 30.0 % |
  | 3 | 14.7 % |

  So a **clean cluster is expected**, plus clusters for the kernels not seen in the showcase (`gaussian_blur` lowers `laplacian_var`; `speckle` is signal-dependent and texture-sensitive).

- **Clusters mix.** Because up to 3 perturbations stack per frame, the number of discoverable clusters is unknown. HDBSCAN may merge mixed-noise frames into a large catch-all cluster or scatter them into the noise bucket (`-1`); either outcome is informative.

### Interpretation protocol

1. Fit HDBSCAN; report `n_clusters`, `n_noise`, and silhouette.
2. For each cluster, **inverse-transform the standardised centroid** back to original feature units and inspect it feature by feature against the fingerprint table in §5. The centroid is more trustworthy than the thumbnails — a cluster is a region of feature space, not necessarily a single kernel.
3. Pull 3 random thumbnails per cluster to cross-check, but **expect visual heterogeneity**.
4. **Name each cluster by what the centroid says**, not what the thumbnails show. If the centroid picks out high HF energy but doesn't disambiguate `gaussian_noise` from `salt_pepper`, name it `"HF-noise (gauss ∨ sp)"` — a feature-space label — rather than forcing a single-kernel name.

### Extracting centroids and representative images

Two useful summaries for each cluster:
- **Centroid** — the average feature values of the cluster. Compare it feature-by-feature against the §5 fingerprint table to name the cluster.
- **Medoid** — the real frame closest to the centroid. Use it as the representative thumbnail.

Pass `store_centers="both"` to HDBSCAN and sklearn computes both. The code cell below demonstrates the workflow on the showcase data — pick a cluster via `CLUSTER_TO_INSPECT` and re-run.

### Expected caveats you should report

- Our showcase assumed one kernel per frame from a 3-kernel palette; inhouse has up to 3 kernels per frame from a 5-kernel palette. The feature set may need **extension when one axis groups two kernels together** — e.g. `log_laplacian_var` doesn't disambiguate `gaussian_blur` from `contrast` when they co-occur with Gaussian noise. The remedy is to add a feature that lights up for one of the two but not the other.
- If you propose and validate an additional feature, document the physical reasoning *before* measuring whether it improved internal metrics.


In [ ]:
# ---- Pick a cluster to inspect ----
CLUSTER_TO_INSPECT = 1  # change to any id listed below, then re-run this cell
N_EXTRA_THUMBNAILS = 3  # random non-medoid members to show for the visual cross-check

# Refit HDBSCAN with `store_centers="both"` so sklearn computes centroids AND medoids.
hdb_c = HDBSCAN(min_cluster_size=5, min_samples=3, store_centers="both").fit(X)
labels_c = hdb_c.labels_
cluster_ids = sorted(int(k) for k in np.unique(labels_c) if k != -1)
print(f"Found clusters: {cluster_ids} (noise = {int((labels_c == -1).sum())} points)")

# Centroids in original feature units — this is the fingerprint to read against §5.
centroids_orig = scaler.inverse_transform(hdb_c.centroids_)
centroid_df = pd.DataFrame(
    centroids_orig, columns=feature_cols, index=[f"c{k}" for k in cluster_ids]
).round(4)
print("\nCluster centroids (original feature units):")
print(centroid_df)

# Medoids: actual frames whose standardised features are closest to each centroid.
medoids_std = hdb_c.medoids_
rep_idx_per_cluster = {
    k: int(np.argmin(np.linalg.norm(X - medoids_std[i], axis=1)))
    for i, k in enumerate(cluster_ids)
}

# ---- Display the chosen cluster ----
assert (
    CLUSTER_TO_INSPECT in cluster_ids
), f"Cluster {CLUSTER_TO_INSPECT} not in {cluster_ids}"
k = CLUSTER_TO_INSPECT
members = np.where(labels_c == k)[0]
rep_idx = rep_idx_per_cluster[k]
rng_vis = np.random.default_rng(SEED + k)
extras = rng_vis.choice(
    [i for i in members if i != rep_idx],
    size=min(N_EXTRA_THUMBNAILS, len(members) - 1),
    replace=False,
)

print(f"\n--- Cluster c{k}: {len(members)} members ---")
print("Centroid (feature → value):")
print(centroid_df.loc[f"c{k}"])
print(
    f"Medoid frame: {sample_paths[rep_idx].name} (true label: {df.loc[rep_idx, 'noise_type']})"
)

cols = 1 + len(extras)
fig, axes = plt.subplots(1, cols, figsize=(3.5 * cols, 4))
axes = np.atleast_1d(axes)
axes[0].imshow(images[rep_idx], cmap="gray", vmin=0, vmax=1)
axes[0].set_title(f"medoid\n{df.loc[rep_idx, 'noise_type']}")
axes[0].axis("off")
for ax, idx in zip(axes[1:], extras):
    ax.imshow(images[int(idx)], cmap="gray", vmin=0, vmax=1)
    ax.set_title(f"extra\n{df.loc[int(idx), 'noise_type']}")
    ax.axis("off")
plt.suptitle(f"Cluster c{k} — representative + {len(extras)} random members", y=1.02)
plt.tight_layout()
plt.show()